In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load the CSV file
file_path = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)

# Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

# Ensure 'Balls' and 'Strikes' columns are numeric
df['Balls'] = pd.to_numeric(df['Balls'], errors='coerce')
df['Strikes'] = pd.to_numeric(df['Strikes'], errors='coerce')

# Add a 'PitcherHand' column based on 'Relside'
def determine_pitcher_hand(rel_side):
    if rel_side > 0:
        return 'R'
    elif rel_side < 0:
        return 'L'
    else:
        return 'Unknown'

df['Pitcherhand'] = df['Relside'].apply(determine_pitcher_hand)

# Define pitch categories based on initial pitch types
pitch_categories = {
    "Breaking Ball": ["Slider", "Curveball"],
    "Fastball": ["Fastball", "Four-Seam", "Sinker", "Cutter"],
    "Offspeed": ["ChangeUp", "Splitter"]
}

# Function to categorize pitch types into broader groups
def categorize_pitch_type(pitch_type):
    for category, pitches in pitch_categories.items():
        if pitch_type in pitches:
            return category
    return "Other"

# Create a new column 'Pitchcategory' to categorize pitches
df['Pitchcategory'] = df['Autopitchtype'].apply(categorize_pitch_type)

# Create boolean columns for each count category
df['FirstPitch'] = (df['Balls'] == 0) & (df['Strikes'] == 0)
df['TwoStrike'] = df['Strikes'] == 2
df['ThreeBall'] = df['Balls'] == 3
df['EvenCount'] = (df['Balls'] == df['Strikes']) & (df['Balls'] != 0)
df['HitterFriendly'] = df['Balls'] > df['Strikes']
df['PitcherFriendly'] = df['Strikes'] > df['Balls']

# Streamlit Sidebar Filters
st.sidebar.header("Filter Options")

# Higher-level pitch categories
pitch_categories_list = list(pitch_categories.keys())
if 'Other' in df['Pitchcategory'].unique():
    pitch_categories_list.append('Other')

selected_categories = st.sidebar.multiselect("Select Pitch Category(s)", pitch_categories_list, default=pitch_categories_list)

# Get the list of specific pitch types in the selected categories
available_pitch_types = []
for category in selected_categories:
    if category in pitch_categories:
        available_pitch_types.extend(pitch_categories[category])
    else:
        # For 'Other' category, get the pitch types not in any category
        categorized_pitches = [pitch for pitches in pitch_categories.values() for pitch in pitches]
        other_pitches = df[~df['Autopitchtype'].isin(categorized_pitches)]['Autopitchtype'].unique().tolist()
        available_pitch_types.extend(other_pitches)

# Remove duplicates and sort the pitch types
available_pitch_types = sorted(list(set(available_pitch_types)))

selected_pitch_types = st.sidebar.multiselect("Select Pitch Type(s)", available_pitch_types, default=available_pitch_types)

pitcher_hands = ['R', 'L']
selected_pitcher_hand = st.sidebar.selectbox("Select Pitcher Hand", pitcher_hands)

# Count filter options
count_options = ['1st-pitch', '2-Strike', '3-Ball', 'Even', 'Hitter-Friendly', 'Pitcher-Friendly']
selected_counts = st.sidebar.multiselect("Select Count(s)", count_options, default=['1st-pitch'])

# Map count options to boolean column names
count_option_to_column = {
    '1st-pitch': 'FirstPitch',
    '2-Strike': 'TwoStrike',
    '3-Ball': 'ThreeBall',
    'Even': 'EvenCount',
    'Hitter-Friendly': 'HitterFriendly',
    'Pitcher-Friendly': 'PitcherFriendly'
}

# Placeholder for Heart/Shadow/Chase filter
# def categorize_plate_location(row):
#     # Function to assign each observation as 'Heart', 'Shadow', 'Chase', etc. based on location
#     pass
# df['PlateZone'] = df.apply(categorize_plate_location, axis=1)
# plate_zones = df['PlateZone'].unique()
# selected_plate_zones = st.sidebar.multiselect("Select Plate Zone(s)", plate_zones, default=plate_zones)

# Filter data based on selection
filtered_data = df[
    (df['Pitchcategory'].isin(selected_categories)) &
    (df['Autopitchtype'].isin(selected_pitch_types)) &
    (df['Pitcherhand'] == selected_pitcher_hand)
    # & (df['PlateZone'].isin(selected_plate_zones))  # Uncomment when PlateZone is defined
]

# Apply count filters
if selected_counts:
    # Create a mask where any of the selected count categories are True
    count_masks = [filtered_data[count_option_to_column[count]] for count in selected_counts]
    combined_mask = np.logical_or.reduce(count_masks)
    filtered_data = filtered_data[combined_mask]
else:
    st.warning("Please select at least one count category.")

# Function to create heatmaps
def create_heatmap(data, metric, ax):
    if data.empty or metric not in data.columns:
        ax.set_title(f"No data available for {metric}.")
        ax.axis('off')
        return

    # Define the strike zone boundaries
    x_min, x_max = -2.5, 2.5
    y_min, y_max = 0, 5

    # Create 2D histogram bins
    x_bins = np.linspace(x_min, x_max, 50)
    y_bins = np.linspace(y_min, y_max, 50)

    # Create a pivot table for the heatmap
    heatmap_data, xedges, yedges = np.histogram2d(
        data['Platelocside'],
        data['Platelocheight'],
        bins=[x_bins, y_bins],
        weights=data[metric],
        density=False
    )

    # Normalize the heatmap data
    counts, _, _ = np.histogram2d(
        data['Platelocside'],
        data['Platelocheight'],
        bins=[x_bins, y_bins]
    )
    heatmap_data = np.divide(heatmap_data, counts, out=np.zeros_like(heatmap_data), where=counts != 0)

    # Transpose heatmap_data for correct orientation
    heatmap_data = heatmap_data.T

    # Plot the heatmap
    extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
    sns.heatmap(
        heatmap_data,
        cmap="coolwarm",
        ax=ax,
        cbar=True,
        center=0,
        extent=extent,
        origin='lower'
    )

    # Draw the strike zone rectangle
    ax.add_patch(plt.Rectangle((-0.83, 1.5), 1.66, 2.1, edgecolor='black', facecolor='none'))

    # Set plot limits
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    ax.set_title(metric)

# Main Page Content
st.title("Hitter Heatmaps")

# Create subplots for the heatmaps
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Heatmap for Launch Angle
if 'Angle' in filtered_data.columns and not filtered_data['Angle'].isnull().all():
    create_heatmap(filtered_data, 'Angle', axs[0])
else:
    axs[0].set_title("Launch Angle")
    axs[0].axis('off')
    axs[0].text(0.5, 0.5, "Launch Angle Heatmap\n(Data Not Available)", horizontalalignment='center', verticalalignment='center')

# Heatmap for Exit Velocity
if 'Exitspeed' in filtered_data.columns and not filtered_data['Exitspeed'].isnull().all():
    create_heatmap(filtered_data, 'Exitspeed', axs[1])
else:
    axs[1].set_title("Exit Velocity")
    axs[1].axis('off')
    axs[1].text(0.5, 0.5, "Exit Velocity Heatmap\n(Data Not Available)", horizontalalignment='center', verticalalignment='center')

# Adjust layout
plt.tight_layout()
st.pyplot(fig)


In [ ]:
##v2

import streamlit as st
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb

# Load the CSV file
file_path = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)

# Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

# Ensure 'Balls' and 'Strikes' columns are numeric
df['Balls'] = pd.to_numeric(df['Balls'], errors='coerce')
df['Strikes'] = pd.to_numeric(df['Strikes'], errors='coerce')

# Add a 'PitcherHand' column based on 'Relside'
def determine_pitcher_hand(rel_side):
    if rel_side > 0:
        return 'R'
    elif rel_side < 0:
        return 'L'
    else:
        return 'Unknown'

df['Pitcherhand'] = df['Relside'].apply(determine_pitcher_hand)

# Define pitch categories based on initial pitch types
pitch_categories = {
    "Breaking Ball": ["Slider", "Curveball"],
    "Fastball": ["Fastball", "Four-Seam", "Sinker", "Cutter"],
    "Offspeed": ["ChangeUp", "Splitter"]
}

# Function to categorize pitch types into broader groups
def categorize_pitch_type(pitch_type):
    for category, pitches in pitch_categories.items():
        if pitch_type in pitches:
            return category
    return "Other"

# Create a new column 'Pitchcategory' to categorize pitches
df['Pitchcategory'] = df['Autopitchtype'].apply(categorize_pitch_type)

# Create boolean columns for each count category
df['FirstPitch'] = (df['Balls'] == 0) & (df['Strikes'] == 0)
df['TwoStrike'] = df['Strikes'] == 2
df['ThreeBall'] = df['Balls'] == 3
df['EvenCount'] = (df['Balls'] == df['Strikes']) & (df['Balls'] != 0)
df['HitterFriendly'] = df['Balls'] > df['Strikes']
df['PitcherFriendly'] = df['Strikes'] > df['Balls']

# Load the saved XGBoost model
xgb_model = xgb.XGBRegressor()
xgb_model.load_model('xgboost_model.json')

# Ensure 'Exitspeed' and 'Angle' are numeric
df['Exitspeed'] = pd.to_numeric(df['Exitspeed'], errors='coerce')
df['Angle'] = pd.to_numeric(df['Angle'], errors='coerce')

# Create a mask where 'Exitspeed' and 'Angle' are not NaN
mask = df['Exitspeed'].notna() & df['Angle'].notna()

# Create the features DataFrame, renaming columns to match model's expectations
X = df.loc[mask, ['Exitspeed', 'Angle']].rename(columns={'Exitspeed': 'launch_speed', 'Angle': 'launch_angle'})

# Apply the model to predict 'PredictedSLG'
df.loc[mask, 'PredictedSLG'] = xgb_model.predict(X)

# Streamlit Sidebar Filters
st.sidebar.header("Filter Options")

# Higher-level pitch categories
pitch_categories_list = list(pitch_categories.keys())
if 'Other' in df['Pitchcategory'].unique():
    pitch_categories_list.append('Other')

selected_categories = st.sidebar.multiselect("Select Pitch Category(s)", pitch_categories_list, default=pitch_categories_list)

# Get the list of specific pitch types in the selected categories
available_pitch_types = []
for category in selected_categories:
    if category in pitch_categories:
        available_pitch_types.extend(pitch_categories[category])
    else:
        # For 'Other' category, get the pitch types not in any category
        categorized_pitches = [pitch for pitches in pitch_categories.values() for pitch in pitches]
        other_pitches = df[~df['Autopitchtype'].isin(categorized_pitches)]['Autopitchtype'].unique().tolist()
        available_pitch_types.extend(other_pitches)

# Remove duplicates and sort the pitch types
available_pitch_types = sorted(list(set(available_pitch_types)))

selected_pitch_types = st.sidebar.multiselect("Select Pitch Type(s)", available_pitch_types, default=available_pitch_types)

pitcher_hands = ['R', 'L']
selected_pitcher_hand = st.sidebar.selectbox("Select Pitcher Hand", pitcher_hands)

# Count filter options
count_options = ['1st-pitch', '2-Strike', '3-Ball', 'Even', 'Hitter-Friendly', 'Pitcher-Friendly']
selected_counts = st.sidebar.multiselect("Select Count(s)", count_options, default=['1st-pitch'])

# Map count options to boolean column names
count_option_to_column = {
    '1st-pitch': 'FirstPitch',
    '2-Strike': 'TwoStrike',
    '3-Ball': 'ThreeBall',
    'Even': 'EvenCount',
    'Hitter-Friendly': 'HitterFriendly',
    'Pitcher-Friendly': 'PitcherFriendly'
}

# Placeholder for Heart/Shadow/Chase filter
# def categorize_plate_location(row):
#     # Function to assign each observation as 'Heart', 'Shadow', 'Chase', etc. based on location
#     pass
# df['PlateZone'] = df.apply(categorize_plate_location, axis=1)
# plate_zones = df['PlateZone'].unique()
# selected_plate_zones = st.sidebar.multiselect("Select Plate Zone(s)", plate_zones, default=plate_zones)

# Filter data based on selection
filtered_data = df[
    (df['Pitchcategory'].isin(selected_categories)) &
    (df['Autopitchtype'].isin(selected_pitch_types)) &
    (df['Pitcherhand'] == selected_pitcher_hand)
    # & (df['PlateZone'].isin(selected_plate_zones))  # Uncomment when PlateZone is defined
]

# Apply count filters
if selected_counts:
    # Create a mask where any of the selected count categories are True
    count_masks = [filtered_data[count_option_to_column[count]] for count in selected_counts]
    combined_mask = np.logical_or.reduce(count_masks)
    filtered_data = filtered_data[combined_mask]
else:
    st.warning("Please select at least one count category.")

# Function to create heatmaps
def create_heatmap(data, metric, ax):
    if data.empty or metric not in data.columns:
        ax.set_title(f"No data available for {metric}.")
        ax.axis('off')
        return

    # Define the strike zone boundaries
    x_min, x_max = -2.5, 2.5
    y_min, y_max = 0, 5

    # Create 2D histogram bins
    x_bins = np.linspace(x_min, x_max, 50)
    y_bins = np.linspace(y_min, y_max, 50)

    # Create a pivot table for the heatmap
    heatmap_data, xedges, yedges = np.histogram2d(
        data['Platelocside'],
        data['Platelocheight'],
        bins=[x_bins, y_bins],
        weights=data[metric],
        density=False
    )

    # Normalize the heatmap data
    counts, _, _ = np.histogram2d(
        data['Platelocside'],
        data['Platelocheight'],
        bins=[x_bins, y_bins]
    )
    heatmap_data = np.divide(heatmap_data, counts, out=np.zeros_like(heatmap_data), where=counts != 0)

    # Transpose heatmap_data for correct orientation
    heatmap_data = heatmap_data.T

    # Plot the heatmap
    extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
    sns.heatmap(
        heatmap_data,
        cmap="coolwarm",
        ax=ax,
        cbar=True,
        center=0,
        extent=extent,
        origin='lower'
    )

    # Draw the strike zone rectangle
    ax.add_patch(plt.Rectangle((-0.83, 1.5), 1.66, 2.1, edgecolor='black', facecolor='none'))

    # Set plot limits
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    ax.set_title(metric)

# Main Page Content
st.title("Hitter Heatmaps")

# Create subplots for the heatmaps
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

# Heatmap for Launch Angle
if 'Angle' in filtered_data.columns and not filtered_data['Angle'].isnull().all():
    create_heatmap(filtered_data, 'Angle', axs[0])
else:
    axs[0].set_title("Launch Angle")
    axs[0].axis('off')
    axs[0].text(0.5, 0.5, "Launch Angle Heatmap\n(Data Not Available)", horizontalalignment='center', verticalalignment='center')

# Heatmap for Exit Velocity
if 'Exitspeed' in filtered_data.columns and not filtered_data['Exitspeed'].isnull().all():
    create_heatmap(filtered_data, 'Exitspeed', axs[1])
else:
    axs[1].set_title("Exit Velocity")
    axs[1].axis('off')
    axs[1].text(0.5, 0.5, "Exit Velocity Heatmap\n(Data Not Available)", horizontalalignment='center', verticalalignment='center')

# Heatmap for Predicted SLG
if 'PredictedSLG' in filtered_data.columns and not filtered_data['PredictedSLG'].isnull().all():
    create_heatmap(filtered_data, 'PredictedSLG', axs[2])
else:
    axs[2].set_title("xSLG")
    axs[2].axis('off')
    axs[2].text(0.5, 0.5, "xSLG Heatmap\n(Data Not Available)", horizontalalignment='center', verticalalignment='center')

# Adjust layout
plt.tight_layout()
st.pyplot(fig)
